# Evaluate Qwen3 1.7B LoRA Adapters

Evaluate the active formula-direct and algorithmic-scaffold adapters on the frozen test split.


In [ ]:
!pip install -q -U mlx-lm pandas matplotlib tqdm


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from mlx_lm import generate, load
from tqdm.auto import tqdm


In [ ]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
import importlib
import training_eval.eval_utils as eval_utils

importlib.reload(eval_utils)

GRADING_POLICY = eval_utils.GRADING_POLICY
default_test_dir = eval_utils.default_test_dir
extract_answer = eval_utils.extract_answer
extract_json_object = eval_utils.extract_json_object
is_correct = eval_utils.is_correct
load_jsonl_records = eval_utils.load_jsonl_records
rows_to_frame = eval_utils.rows_to_frame
save_results = eval_utils.save_results
summarize_accuracy = eval_utils.summarize_accuracy


In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B-MLX-bf16"
ADAPTER_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora"
RESULT_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora_eval"
MAX_NEW_TOKENS = 512

In [ ]:
ADAPTERS = {
    "formula_direct": ADAPTER_ROOT / "formula_direct" / "adapters",
    "algorithmic_scaffold_v2": ADAPTER_ROOT / "algorithmic_scaffold_v2" / "adapters",
    "algorithmic_scaffold_v3_1_lora8": ADAPTER_ROOT / "algorithmic_scaffold_v3_1_lora8" / "adapters",
    "algorithmic_scaffold_v3_5": ADAPTER_ROOT / "algorithmic_scaffold_v3_5" / "adapters",
    "algorithmic_scaffold_v3_5_lora12": ADAPTER_ROOT / "algorithmic_scaffold_v3_5_lora12" / "adapters",
}

# Keep this list focused: these are the adapters worth reporting on the frozen test set.
ADAPTERS_TO_EVALUATE = [
    "algorithmic_scaffold_v2",
    "algorithmic_scaffold_v3_1_lora8",
    "algorithmic_scaffold_v3_5",
    "algorithmic_scaffold_v3_5_lora12",
]
ACTIVE_ADAPTERS = {name: ADAPTERS[name] for name in ADAPTERS_TO_EVALUATE}


In [ ]:
records = load_jsonl_records(default_test_dir(PROJECT_ROOT), pattern="*_preview.jsonl")
len(records)

In [ ]:
FINE_TUNE_SYSTEM_MESSAGE = 'You solve discrete stochastic-process problems. Give concise reasoning, then end with exactly one final answer block: Final answer:\n<answer>\n{...}\n</answer>. Do not write anything after </answer>.'


In [ ]:
def make_fine_tuned_chat_messages(problem):
    return [
        {"role": "system", "content": FINE_TUNE_SYSTEM_MESSAGE},
        {"role": "user", "content": problem},
    ]


In [ ]:
def make_qwen_prompt(tokenizer, problem):
    messages = make_fine_tuned_chat_messages(problem)
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
def load_adapter_model(adapter_path):
    return load(MODEL_NAME, adapter_path=str(adapter_path))

In [ ]:
def generate_answer(model, tokenizer, problem):
    prompt = make_qwen_prompt(tokenizer, problem)
    return generate(model, tokenizer, prompt=prompt, max_tokens=MAX_NEW_TOKENS, verbose=False)

In [ ]:
def evaluate_adapter(adapter_label, adapter_path, records):
    model, tokenizer = load_adapter_model(adapter_path)
    rows = []

    for record in tqdm(records, desc=adapter_label):
        raw_output = generate_answer(model, tokenizer, record["problem"])
        predicted = extract_answer(raw_output, record["canonical_answer"])
        metadata = record.get("metadata", {})

        rows.append({
            "adapter": adapter_label,
            "id": record["id"],
            "family": record["family"],
            "problem_type": record["problem_type"],
            "difficulty": record["difficulty"],
            "answer_type": record["answer_type"],
            "manual_variation": metadata.get("manual_variation", False),
            "problem": record["problem"],
            "canonical_answer": record["canonical_answer"],
            "raw_output": raw_output,
            "predicted_answer": predicted,
            "correct": is_correct(predicted, record["canonical_answer"]),
        })

    return rows

In [ ]:
def save_adapter_eval(adapter_label, rows):
    df = rows_to_frame(rows)
    metrics = summarize_accuracy(df)
    metrics.update({
        "model": MODEL_NAME,
        "adapter": adapter_label,
        "dataset": "benchmark/data/test/*_preview.jsonl",
        "grading_policy": GRADING_POLICY,
    })
    return save_results(rows, RESULT_ROOT / adapter_label, metrics)

In [ ]:
all_rows = []

for adapter_label, adapter_path in ACTIVE_ADAPTERS.items():
    rows = evaluate_adapter(adapter_label, adapter_path, records)
    save_adapter_eval(adapter_label, rows)
    all_rows.extend(rows)

df = rows_to_frame(all_rows)
df.head()


In [ ]:
display(df.groupby("adapter")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby(["adapter", "answer_type"])["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby(["adapter", "family"])["correct"].agg(["mean", "sum", "count"]).sort_index())

In [ ]:
df.loc[~df["correct"], ["adapter", "id", "family", "problem_type", "canonical_answer", "predicted_answer", "raw_output"]].head(30)

## Compare Against Baselines

Load saved baseline outputs and regrade them with the current evaluator. The comparison tables deliberately split binary and non-binary questions; overall accuracy is useful only as a quick headline.


In [ ]:
BASELINE_RESULT_DIRS = {
    "Qwen3 1.7B base": PROJECT_ROOT / "results" / "baselines" / "qwen3_1_7b_test_closed_book",
    "Qwen3 8B API": PROJECT_ROOT / "results" / "baselines" / "qwen_qwen3_8b_test_closed_book_api",
    "Qwen3 32B API": PROJECT_ROOT / "results" / "baselines" / "qwen_qwen3_32b_test_closed_book_api",
    "Qwen3 235B-A22B API": PROJECT_ROOT / "results" / "baselines" / "qwen_qwen3_235b_a22b_test_closed_book_api",
    "DeepSeek V4 Flash API": PROJECT_ROOT / "results" / "baselines" / "deepseek_deepseek_v4_flash_free_test_closed_book_api",
    "GPT OSS 20B API": PROJECT_ROOT / "results" / "baselines" / "openai_gpt_oss_20b_free_test_closed_book_api",
}


In [ ]:
def load_result_rows(label, result_dir, kind):
    path = result_dir / "outputs.jsonl"
    if not path.exists():
        return []

    rows = []
    with path.open() as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            canonical = row["canonical_answer"]
            raw_output = row.get("raw_output", "")
            predicted = extract_answer(raw_output, canonical)
            rows.append({
                "run": label,
                "kind": kind,
                "id": row["id"],
                "family": row["family"],
                "problem_type": row["problem_type"],
                "difficulty": row["difficulty"],
                "answer_type": row["answer_type"],
                "canonical_answer": canonical,
                "raw_output": raw_output,
                "predicted_answer": predicted,
                "correct": is_correct(predicted, canonical),
            })
    return rows


In [ ]:
baseline_rows = []
for label, result_dir in BASELINE_RESULT_DIRS.items():
    baseline_rows.extend(load_result_rows(label, result_dir, kind="baseline"))

adapter_rows = []
for adapter_label in ADAPTERS_TO_EVALUATE:
    adapter_rows.extend(load_result_rows(
        f"Qwen3 1.7B LoRA {adapter_label}",
        RESULT_ROOT / adapter_label,
        kind="fine_tuned",
    ))

comparison_df = rows_to_frame(baseline_rows + adapter_rows)
display(comparison_df.groupby(["kind", "run", "answer_type"]).size().rename("count").reset_index())
comparison_df.head()


In [ ]:
def summarize_run(group):
    non_biased = group[group["problem_type"] != "biased_boundaries_zero_a"]
    binary = group[group["answer_type"] == "binary"]
    non_binary = group[group["answer_type"] == "non_binary"]
    non_binary_no_biased = non_binary[non_binary["problem_type"] != "biased_boundaries_zero_a"]
    return pd.Series({
        "accuracy": group["correct"].mean(),
        "correct": int(group["correct"].sum()),
        "count": int(len(group)),
        "binary_accuracy": binary["correct"].mean() if len(binary) else None,
        "binary_correct": int(binary["correct"].sum()),
        "binary_count": int(len(binary)),
        "non_binary_accuracy": non_binary["correct"].mean() if len(non_binary) else None,
        "non_binary_correct": int(non_binary["correct"].sum()),
        "non_binary_count": int(len(non_binary)),
        "exclude_biased_accuracy": non_biased["correct"].mean() if len(non_biased) else None,
        "non_binary_exclude_biased_accuracy": non_binary_no_biased["correct"].mean() if len(non_binary_no_biased) else None,
    })


In [ ]:
summary_df = (
    comparison_df
    .groupby(["kind", "run"])
    .apply(summarize_run)
    .reset_index()
    .sort_values(["accuracy", "exclude_biased_accuracy"], ascending=False)
)

answer_type_summary_df = (
    comparison_df
    .groupby(["kind", "run", "answer_type"])["correct"]
    .agg(accuracy="mean", correct="sum", count="count")
    .reset_index()
    .sort_values(["answer_type", "accuracy"], ascending=[True, False])
)

display(answer_type_summary_df)
summary_df


In [ ]:
subclass_df = (
    comparison_df
    .groupby(["kind", "run", "answer_type", "problem_type"])["correct"]
    .agg(accuracy="mean", correct="sum", count="count")
    .reset_index()
    .sort_values(["answer_type", "problem_type", "accuracy"], ascending=[True, True, False])
)

family_df = (
    comparison_df
    .groupby(["kind", "run", "answer_type", "family"])["correct"]
    .agg(accuracy="mean", correct="sum", count="count")
    .reset_index()
    .sort_values(["answer_type", "family", "accuracy"], ascending=[True, True, False])
)

difficulty_df = (
    comparison_df
    .groupby(["kind", "run", "answer_type", "difficulty"])["correct"]
    .agg(accuracy="mean", correct="sum", count="count")
    .reset_index()
    .sort_values(["answer_type", "difficulty", "accuracy"], ascending=[True, True, False])
)

display(subclass_df)
display(family_df)
difficulty_df


## Comparison Charts

These plots keep binary and non-binary tasks visibly separated. They are saved under the comparison results folder.


In [ ]:
comparison_dir = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora_eval_comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)

RUN_ORDER = (
    summary_df
    .sort_values(["kind", "accuracy"], ascending=[True, False])["run"]
    .tolist()
)


In [ ]:
def plot_answer_type_accuracy(answer_type_summary_df):
    plot_df = answer_type_summary_df.pivot(index="run", columns="answer_type", values="accuracy")
    plot_df = plot_df.reindex([run for run in RUN_ORDER if run in plot_df.index])

    ax = plot_df.plot(kind="bar", figsize=(12, 5), ylim=(0, 1), width=0.82)
    ax.set_title("Accuracy by model and answer type")
    ax.set_xlabel("")
    ax.set_ylabel("Accuracy")
    ax.axhline(0.5, color="0.75", linestyle="--", linewidth=1)
    ax.legend(title="Answer type")
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.savefig(comparison_dir / "accuracy_by_answer_type.png", dpi=180)
    plt.show()


In [ ]:
plot_answer_type_accuracy(answer_type_summary_df)


In [ ]:
def plot_subclass_heatmap(subclass_df, answer_type):
    plot_df = subclass_df[subclass_df["answer_type"] == answer_type]
    matrix = plot_df.pivot(index="problem_type", columns="run", values="accuracy")
    matrix = matrix[[run for run in RUN_ORDER if run in matrix.columns]]

    fig_width = max(8, 0.85 * len(matrix.columns))
    fig_height = max(4, 0.38 * len(matrix.index))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    image = ax.imshow(matrix.fillna(0), vmin=0, vmax=1, aspect="auto", cmap="RdYlGn")
    ax.set_title(f"{answer_type.replace('_', '-').title()} accuracy by subclass")
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=35, ha="right")
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)

    for i, problem_type in enumerate(matrix.index):
        for j, run in enumerate(matrix.columns):
            value = matrix.loc[problem_type, run]
            if pd.notna(value):
                ax.text(j, i, f"{value:.0%}", ha="center", va="center", fontsize=8)

    fig.colorbar(image, ax=ax, label="Accuracy")
    plt.tight_layout()
    plt.savefig(comparison_dir / f"accuracy_by_subclass_{answer_type}.png", dpi=180)
    plt.show()


In [ ]:
plot_subclass_heatmap(subclass_df, "binary")
plot_subclass_heatmap(subclass_df, "non_binary")


In [ ]:
def plot_split_grouped_accuracy(group_df, group_column, filename):
    for answer_type in sorted(group_df["answer_type"].dropna().unique()):
        plot_df = group_df[group_df["answer_type"] == answer_type]
        matrix = plot_df.pivot(index=group_column, columns="run", values="accuracy")
        matrix = matrix[[run for run in RUN_ORDER if run in matrix.columns]]

        ax = matrix.plot(kind="bar", figsize=(12, 4.8), ylim=(0, 1), width=0.82)
        ax.set_title(f"{answer_type.replace('_', '-').title()} accuracy by {group_column}")
        ax.set_xlabel(group_column)
        ax.set_ylabel("Accuracy")
        ax.axhline(0.5, color="0.75", linestyle="--", linewidth=1)
        ax.legend(title="Run", bbox_to_anchor=(1.02, 1), loc="upper left")
        ax.tick_params(axis="x", rotation=25)
        plt.tight_layout()
        plt.savefig(comparison_dir / f"{filename}_{answer_type}.png", dpi=180)
        plt.show()


In [ ]:
plot_split_grouped_accuracy(family_df, "family", "accuracy_by_family")
plot_split_grouped_accuracy(difficulty_df, "difficulty", "accuracy_by_difficulty")


In [ ]:
comparison_dir = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora_eval_comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(comparison_dir / "summary.csv", index=False)
answer_type_summary_df.to_csv(comparison_dir / "by_answer_type.csv", index=False)
subclass_df.to_csv(comparison_dir / "by_subclass.csv", index=False)
family_df.to_csv(comparison_dir / "by_family.csv", index=False)
difficulty_df.to_csv(comparison_dir / "by_difficulty.csv", index=False)
comparison_df.to_csv(comparison_dir / "all_outputs_regraded.csv", index=False)
comparison_dir


In [ ]:
comparison_df.loc[
    ~comparison_df["correct"],
    ["kind", "run", "id", "family", "problem_type", "canonical_answer", "predicted_answer", "raw_output"]
].head(40)
